# 02 · Random forest on the anomaly target (main grid search)

The forest predicts how far each season is from the station's usual date:

- **station reference** = the station's mean full-bloom DOY over the last 30 years of the fold's training data
  (stations with fewer than 10 records use the region mean, then the overall mean);
- **target** = full-bloom DOY − station reference;
- the reference is added back before scoring, so all scores are in days of full-bloom DOY;
- the model trains on **all** training years of each fold (the 30 years are only for the reference).

| | |
|---|---|
| Datasets | Mar15 / Mar1 / Feb1 × `large` (82 stations) / `small` (48 stations) |
| Grid | 54 candidates: `n_estimators` 200/500 · `max_depth` 5/10/None · `min_samples_leaf` 1/10/30 · `max_features` sqrt/0.7/1.0 |
| Output | `results/rf/02_anomaly_grid/` — `{date}_{sample}_*.csv` and `best_models.csv` (one row per dataset, used by `src/models/log_models.py`) |

**Status: results in the repo come from the original run (not rerun in the clean-up).**
A full run is 54 × 5 × 6 = 1,620 fits (about 1–2 hours on 2 CPUs). Use `MAX_CANDIDATES` / `OUTPUT_OVERRIDE` to test.
The 15 March small-sample search is refined further in notebook 03.

In [ ]:
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid

from rf_common import *

RUN_SAMPLES = ['large', 'small']   # run one or both samples
MAX_CANDIDATES = None              # None = full grid; e.g. 2 = quick test
OUTPUT_OVERRIDE = None             # None = results/rf/02_anomaly_grid
N_JOBS = 2

OUT = output_dir('02_anomaly_grid', OUTPUT_OVERRIDE)
print('Results go to:', OUT)

## 1. Datasets and grid

In [ ]:
datasets = {
    dataset_name(date, sample): load_sample(date, sample)
    for sample in RUN_SAMPLES
    for date in ['mar15', 'mar1', 'feb1']
}

param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 10, 30],
    'max_features': ['sqrt', 0.7, 1.0],
}
candidates = list(ParameterGrid(param_grid))[:MAX_CANDIDATES]

display(pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'stations': df['station_id'].nunique(),
     'features': len(feature_columns(df))}
    for name, df in datasets.items()
]))
print(f'{len(datasets)} datasets × {len(candidates)} candidates × 5 folds = '
      f'{len(datasets) * len(candidates) * 5} fits')

## 2. Grid search

Each dataset is saved as soon as it finishes. `best_models.csv` is updated row by row, so running only one sample
keeps the other sample's rows.

In [ ]:
best_models_path = OUT / 'best_models.csv'
comparison_rows = {}
if best_models_path.exists():
    for row in read_csv(best_models_path).to_dict('records'):
        comparison_rows[row['dataset']] = row

for name, df in datasets.items():
    features = feature_columns(df)
    date_label, sample = name.split('-')
    prefix = f'{DATES[date_label]}_{sample}'

    # Folds and station references are the same for every candidate.
    folds, period_rows = [], []
    for fold, train, valid in cv_folds(df):
        check_folds(train, valid, f'{name} fold {fold}')
        reference = make_reference(train)
        train_reference = reference_values(reference, train)
        folds.append({
            'fold': fold,
            'X_train': train[features],
            'X_valid': valid[features],
            'y_train': train[TARGET].to_numpy(dtype=float),
            'y_valid': valid[TARGET].to_numpy(dtype=float),
            'train_reference': train_reference,
            'valid_reference': reference_values(reference, valid),
        })
        period_rows.append({
            'fold': fold,
            'train_years': f"{train['bloom_year'].min()}–{train['bloom_year'].max()}",
            'reference_years': f"{reference['start']}–{reference['end']}",
            'valid_years': f"{valid['bloom_year'].min()}–{valid['bloom_year'].max()}",
            'train_rows': len(train),
            'valid_rows': len(valid),
        })

    display(Markdown(f'### {name}'))
    display(pd.DataFrame(period_rows))

    fold_rows = []
    for candidate_id, params in enumerate(candidates):
        for item in folds:
            model = RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=N_JOBS)
            model.fit(item['X_train'], item['y_train'] - item['train_reference'])

            train_scores = evaluate(item['y_train'],
                                    model.predict(item['X_train']) + item['train_reference'])
            valid_scores = evaluate(item['y_valid'],
                                    model.predict(item['X_valid']) + item['valid_reference'])
            fold_rows.append({
                'candidate_id': candidate_id,
                'fold': item['fold'],
                'train_MAE': train_scores['MAE'],
                'valid_MAE': valid_scores['MAE'],
                'valid_RMSE': valid_scores['RMSE'],
                'valid_R2': valid_scores['R2'],
                'valid_hit_rate_3d': valid_scores['hit_rate_3d'],
            })

    fold_results = pd.DataFrame(fold_rows)
    mean_scores = fold_results.groupby('candidate_id').agg(
        train_MAE=('train_MAE', 'mean'),
        valid_MAE=('valid_MAE', 'mean'),
        valid_MAE_std=('valid_MAE', 'std'),
        valid_RMSE=('valid_RMSE', 'mean'),
        valid_R2=('valid_R2', 'mean'),
        valid_hit_rate_3d=('valid_hit_rate_3d', 'mean'),
    )
    parameter_table = pd.DataFrame(candidates)
    parameter_table.index.name = 'candidate_id'
    gs_results = parameter_table.join(mean_scores).sort_values('valid_MAE', kind='stable')

    best_id = int(gs_results.index[0])
    best_params = candidates[best_id].copy()
    best_fold_results = (
        fold_results.loc[fold_results['candidate_id'] == best_id]
        .sort_values('fold').reset_index(drop=True)
    )

    save_csv(gs_results, OUT / f'{prefix}_grid_search_all_candidates.csv', index=True)
    save_csv(fold_results, OUT / f'{prefix}_all_candidates_fold_results.csv')
    save_csv(best_fold_results, OUT / f'{prefix}_best_candidate_fold_results.csv')

    best = gs_results.loc[best_id]
    comparison_rows[name] = {
        'dataset': name,
        'target_mode': 'anomaly',
        'reference_years': REFERENCE_YEARS,
        'training_window': 'all',
        'features': len(features),
        'best_candidate_id': best_id,
        **best_params,
        **best[['train_MAE', 'valid_MAE', 'valid_MAE_std', 'valid_RMSE',
                'valid_R2', 'valid_hit_rate_3d']].to_dict(),
    }
    order = [dataset_name(d, s) for s in SAMPLES for d in ['mar15', 'mar1', 'feb1']]
    comparison = pd.DataFrame(
        [comparison_rows[key] for key in order if key in comparison_rows]
    )
    save_csv(comparison, best_models_path)

    print(f'Saved {prefix}: best {best_params}, MAE {best["valid_MAE"]:.4f}', flush=True)
    display(gs_results.head(5).round(4))

display(Markdown('### Best model per dataset'))
display(comparison.round(3))